# FinBERT Sentiment Training – Financial PhraseBank & FiQA

**Purpose:** **Fine-tune** ProsusAI/finbert on the canonical FP/FiQA dataset produced by **notebook 03** (same data as 04b). Apples-to-apples comparison: inference-only FinBERT vs **tuned** FinBERT vs TF-IDF+LogReg vs QNLP.

**Flow:** Setup → Load canonical train/val/test from 03_sentiment_FP_FiQA_feature_engineering → Label encoding → Fine-tune FinBERT → Val and test F1 macro → Save model.

**Output:** Fine-tuned model saved to `models/sentiment/finbert_tuned_v1/`. **Dependencies:** Run **03_sentiment_FP_FiQA_feature_engineering.ipynb** first; then `transformers`, `datasets`, `torch`.

**Resume-ready summary (update after re-runs)**  
- **Sample sizes** (from `data/credit_risk_sentiment/processed/meta.json`): train Financial PhraseBank n=799, FiQA n=94 (total train 893); test FP n=101, FiQA n=11 (total test 112).  
- **F1 (test, macro):** FinBERT fine-tuned **0.89** (notebook Section 5); TF-IDF+LogReg **0.76** (04b); inference-only FinBERT **~0.04** (04b).  
- **Bullet:** *Fine-tuned FinBERT on Financial PhraseBank n=799 and FiQA n=94 (F1 0.89 vs TF-IDF+LogReg 0.76 vs inference-only 0.04). Failure-mode fixes: negation handling, conditional/hedged-language penalties, numeric comparison context. Outputs feed into PD feature pipeline (sentiment_score, confidence, flags).*

## 1. Setup: Colab vs local

Run this cell first. On Colab: clone repo, install deps, set ROOT. On local: ensure you're in the repo root.

In [9]:
import os
import sys
from pathlib import Path

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_URL = "https://github.com/leemingloon/ocr-agentic-rag.git"
    REPO_DIR = "ocr-agentic-rag"
    if not os.path.isdir(REPO_DIR):
        get_ipython().system(f"git clone {REPO_URL} {REPO_DIR}")
    get_ipython().run_line_magic("cd", REPO_DIR)
    get_ipython().system("pip install -q transformers datasets accelerate")
    ROOT = Path(".").resolve()
    print("Colab: repo ready. Run the next cells to load data and fine-tune FinBERT.")
else:
    ROOT = Path(".").resolve() if (Path(".").resolve() / "credit_risk").exists() else Path("..").resolve()
    if str(ROOT) not in sys.path:
        sys.path.insert(0, str(ROOT))
    print("Local run. Ensure repo root and install: pip install transformers datasets accelerate.")

Local run. Ensure repo root and install: pip install transformers datasets accelerate.


## 2. Load canonical train/val/test from notebook 03

Load the canonical splits produced by **03_sentiment_FP_FiQA_feature_engineering.ipynb** (`data/credit_risk_sentiment/processed/`). Same splits as 04b for apples-to-apples comparison.

In [10]:
import pandas as pd
import numpy as np

processed_dir = ROOT / "data" / "credit_risk_sentiment" / "processed"
if not (processed_dir / "train.parquet").exists():
    raise FileNotFoundError("Run 03_sentiment_FP_FiQA_feature_engineering.ipynb first to create train/val/test parquets in data/credit_risk_sentiment/processed/")

USE_AUGMENTED_TRAIN = False  # set True and run 03 Section 5b to create train_augmented.parquet first
train_path = processed_dir / "train_augmented.parquet" if (USE_AUGMENTED_TRAIN and (processed_dir / "train_augmented.parquet").exists()) else processed_dir / "train.parquet"
train_df = pd.read_parquet(train_path)
val_df = pd.read_parquet(processed_dir / "val.parquet")
test_df = pd.read_parquet(processed_dir / "test.parquet")
y_train = train_df["label"].values
y_val = val_df["label"].values
y_test = test_df["label"].values
print("Train:", len(train_df), "(augmented)" if USE_AUGMENTED_TRAIN else "", "| Val:", len(val_df), "| Test:", len(test_df))

Train: 893  | Val: 112 | Test: 112


*(Canonical train/val/test from notebook 03; next: label encoding.)*

In [11]:
# Splits loaded from Section 2 (03_sentiment_FP_FiQA_feature_engineering). No in-notebook split.

## 3. Label encoding (match FinBERT id2label)

ProsusAI/finbert uses: 0=negative, 1=neutral, 2=positive. I map string labels to these IDs.

In [12]:
LABEL2ID = {"negative": 0, "neutral": 1, "positive": 2}
ID2LABEL = {0: "negative", 1: "neutral", 2: "positive"}
num_labels = 3

y_train_id = np.array([LABEL2ID.get(str(l).strip().lower(), 1) for l in y_train])
y_val_id = np.array([LABEL2ID.get(str(l).strip().lower(), 1) for l in y_val])
y_test_id = np.array([LABEL2ID.get(str(l).strip().lower(), 1) for l in y_test])

### FinBERT best practices (community & literature)

This notebook follows common practice: Trainer API, F1 macro, 3-class labels. For more (warmup, weight decay, data size, multi-dataset eval), see **`docs/BENCHMARKS_AND_IMPROVEMENTS.md`** — Section 2.

## 4. Fine-tune FinBERT with Hugging Face Trainer

Load tokenizer and model, create a small Dataset, and train with 2–3 epochs and a modest learning rate. I use F1 macro for evaluation so it's comparable to notebook 04b.

In [13]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset
import numpy as np

model_name = "ProsusAI/finbert"
max_length = 128
batch_size = 16
epochs = 3
lr = 2e-5

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)

def tokenize(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=max_length)

train_ds = Dataset.from_dict({
    "text": train_df["text"].astype(str).tolist(),
    "label": y_train_id.tolist(),
})
val_ds = Dataset.from_dict({
    "text": val_df["text"].astype(str).tolist(),
    "label": y_val_id.tolist(),
})
train_ds = train_ds.map(tokenize, batched=True, remove_columns=["text"])
val_ds = val_ds.map(tokenize, batched=True, remove_columns=["text"])
test_ds = Dataset.from_dict({"text": test_df["text"].astype(str).tolist(), "label": y_test_id.tolist()})
test_ds = test_ds.map(tokenize, batched=True, remove_columns=["text"])
test_ds.set_format("torch")
train_ds.set_format("torch")
val_ds.set_format("torch")

def compute_metrics(eval_pred):
    from sklearn.metrics import f1_score
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    f1 = f1_score(labels, preds, average="macro", zero_division=0)
    return {"f1_macro": f1}

# No intermediate checkpoints (save_strategy="no") to save disk; final model saved explicitly below.
training_args = TrainingArguments(
    output_dir=str(ROOT / "models" / "sentiment" / "finbert_tuned_run"),
    num_train_epochs=epochs,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    learning_rate=lr,
    eval_strategy="epoch",
    save_strategy="no",
    warmup_ratio=0.1,
    weight_decay=0.01,
    logging_steps=50,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)
trainer.train()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Map:   0%|          | 0/893 [00:00<?, ? examples/s]

Map:   0%|          | 0/112 [00:00<?, ? examples/s]

Map:   0%|          | 0/112 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

c:\Users\leemi\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=168, training_loss=0.6955978189195905, metrics={'train_runtime': 1594.171, 'train_samples_per_second': 1.68, 'train_steps_per_second': 0.105, 'total_flos': 176220211523328.0, 'train_loss': 0.6955978189195905, 'epoch': 3.0})

## 5. Validation and test F1, save model

Evaluate on **val** and **test** so both numbers are available; you can choose which to report. Save the best model to `models/sentiment/finbert_tuned_v1/`.

In [14]:
# Training sample sizes by source (resume-ready: n=799 PhraseBank, n=94 FiQA)
import json
if "source" in train_df.columns:
    n_fp = (train_df["source"] == "FinancialPhraseBank").sum()
    n_fiqa = (train_df["source"] == "FiQA").sum()
    print("Train: Financial PhraseBank n={}, FiQA n={} (total {})".format(n_fp, n_fiqa, len(train_df)))
else:
    meta_path = processed_dir / "meta.json"
    if meta_path.exists():
        meta = json.load(open(meta_path, encoding="utf-8"))
        n_fp = meta.get("n_train_FinancialPhraseBank", 0)
        n_fiqa = meta.get("n_train_FiQA", 0)
        print("Train: Financial PhraseBank n={}, FiQA n={} (total {}; from meta.json)".format(n_fp, n_fiqa, meta.get("n_train", 0)))
    else:
        print("Train: total n={} (no source or meta; run notebook 03 for per-dataset n)".format(len(train_df)))

eval_out = trainer.evaluate()
f1_finbert_tuned_val = eval_out.get("eval_f1_macro", 0.0)
print("FinBERT (fine-tuned) validation F1 macro:", round(f1_finbert_tuned_val, 4))

test_out = trainer.evaluate(eval_dataset=test_ds)
f1_finbert_tuned_test = test_out.get("eval_f1_macro", 0.0)
print("FinBERT (fine-tuned) test F1 macro:", round(f1_finbert_tuned_test, 4))

# Per-dataset val/test F1 and sample sizes (Financial PhraseBank vs FiQA)
from sklearn.metrics import f1_score
pred_val = trainer.predict(val_ds)
pred_val_id = np.argmax(pred_val.predictions, axis=-1)
pred_test = trainer.predict(test_ds)
pred_test_id = np.argmax(pred_test.predictions, axis=-1)

def _per_source_f1(df, y_true_id, y_pred_id, split_name):
    if "source" not in df.columns:
        return
    for src in ["FinancialPhraseBank", "FiQA"]:
        mask = df["source"] == src
        n = mask.sum()
        if n == 0:
            continue
        f1 = f1_score(y_true_id[mask], y_pred_id[mask], average="macro", zero_division=0)
        print("  {} {} (n={}): F1 macro {:.4f}".format(split_name, src, n, f1))

print("Per-dataset val F1:")
_per_source_f1(val_df, y_val_id, pred_val_id, "Val")
print("Per-dataset test F1:")
_per_source_f1(test_df, y_test_id, pred_test_id, "Test")

save_dir = ROOT / "models" / "sentiment" / "finbert_tuned_v1"
save_dir.mkdir(parents=True, exist_ok=True)
try:
    trainer.save_model(str(save_dir))
    tokenizer.save_pretrained(str(save_dir))
    print("Saved fine-tuned FinBERT to", save_dir)
except Exception as e:
    # Windows error 1224: file with user-mapped section open (safetensors I/O)
    is_safetensor_io = (
        getattr(e, "errno", None) == 1224
        or type(e).__name__ == "SafetensorError"
        or "user-mapped" in str(e).lower()
    )
    if is_safetensor_io:
        # Same path, PyTorch format only (loadable by 04z and rest of repo)
        trainer.model.save_pretrained(str(save_dir), safe_serialization=False)
        tokenizer.save_pretrained(str(save_dir))
        print("Saved fine-tuned FinBERT to", save_dir, "(PyTorch .bin; safetensors failed: Windows file lock)")
    else:
        raise

c:\Users\leemi\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Train: Financial PhraseBank n=799, FiQA n=94 (total 893)


FinBERT (fine-tuned) validation F1 macro: 0.7998


c:\Users\leemi\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


FinBERT (fine-tuned) test F1 macro: 0.8854


c:\Users\leemi\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
c:\Users\leemi\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Per-dataset val F1:
  Val FinancialPhraseBank (n=100): F1 macro 0.8331
  Val FiQA (n=12): F1 macro 0.5940
Per-dataset test F1:
  Test FinancialPhraseBank (n=101): F1 macro 0.9461
  Test FiQA (n=11): F1 macro 0.3385


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved fine-tuned FinBERT to C:\Users\leemi\OneDrive\Desktop\Job_search_2026\ocr-agentic-rag\models\sentiment\finbert_tuned_v1_pytorch (PyTorch .bin; safetensors failed: Windows file lock). Use this path for loading.


### Per-dataset F1 (Financial PhraseBank vs FiQA)

When the canonical data includes a `source` column (from notebook 03), report F1 macro per dataset for resume/documentation.

In [15]:
# Per-dataset F1 (FinBERT fine-tuned) when source is available
from sklearn.metrics import f1_score

def _per_source_f1(df, y_true_id, y_pred_id, split_name="Val"):
    if "source" not in df.columns:
        print("No 'source' column; run 03_sentiment_FP_FiQA_feature_engineering.ipynb and re-load processed data.")
        return
    for src in ["FinancialPhraseBank", "FiQA"]:
        mask = df["source"] == src
        n = mask.sum()
        if n == 0:
            continue
        f1 = f1_score(y_true_id[mask], y_pred_id[mask], average="macro", zero_division=0)
        print(f"  {split_name} {src} (n={n}): F1 macro {f1:.4f}")

# Val
pred_val = trainer.predict(val_ds)
pred_val_id = np.argmax(pred_val.predictions, axis=-1)
print("FinBERT (fine-tuned) per-dataset val F1:")
_per_source_f1(val_df, y_val_id, pred_val_id, "Val")
# Test
pred_test = trainer.predict(test_ds)
pred_test_id = np.argmax(pred_test.predictions, axis=-1)
print("FinBERT (fine-tuned) per-dataset test F1:")
_per_source_f1(test_df, y_test_id, pred_test_id, "Test")

c:\Users\leemi\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


FinBERT (fine-tuned) per-dataset val F1:
  Val FinancialPhraseBank (n=100): F1 macro 0.8331
  Val FiQA (n=12): F1 macro 0.5940


c:\Users\leemi\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


FinBERT (fine-tuned) per-dataset test F1:
  Test FinancialPhraseBank (n=101): F1 macro 0.9461
  Test FiQA (n=11): F1 macro 0.3385


## 6. Failure-mode evaluation (before/after)

Run the **enhanced SentimentPipeline** (negation, conditional, numeric, hedge, ABSA) on the test set and report F1 on the **full** set and on **failure-mode subsets** (negation, conditional, numeric_comparison, hedged). Saves results to `data/proof/credit_risk_sentiment/eval_failure_modes.json` for documentation. Ensures no regression on full test F1.

In [16]:
# Failure-mode evaluation: pipeline with fixes vs baseline (no fixes)
import json
from pathlib import Path
try:
    from credit_risk.sentiment import SentimentPipeline, SentimentPipelineConfig
    from credit_risk.sentiment.evaluation import evaluate_on_subsets, compute_f1_macro, print_evaluation_report
    from credit_risk.sentiment.negation import NegationHandler
    from credit_risk.sentiment.conditional import ConditionalDetector
    from credit_risk.sentiment.numeric_comparison import NumericComparisonExtractor
    from credit_risk.sentiment.hedging import HedgeScorer
    _neg = NegationHandler()
    _cond = ConditionalDetector()
    _num = NumericComparisonExtractor(underperform_pct=5.0)
    _hedge = HedgeScorer()
    texts = test_df["text"].astype(str).tolist()
    y_true = test_df["label"].astype(str).tolist()
    # Baseline: no fixes
    config_baseline = SentimentPipelineConfig(use_negation_handling=False, use_conditional_detection=False, use_numeric_comparison=False, use_hedge_adjustment=False, use_entity_sentiment=False)
    pipe_baseline = SentimentPipeline(config=config_baseline)
    pipe_baseline.config.model_path = str(ROOT / "models" / "sentiment" / "finbert_tuned_v1") if (ROOT / "models" / "sentiment" / "finbert_tuned_v1").exists() else None
    pred_baseline = [pipe_baseline.predict(t)["sentence_sentiment"] for t in texts]
    # With fixes
    config_fixes = SentimentPipelineConfig()
    config_fixes.model_path = str(ROOT / "models" / "sentiment" / "finbert_tuned_v1") if (ROOT / "models" / "sentiment" / "finbert_tuned_v1").exists() else None
    pipe_fixes = SentimentPipeline(config=config_fixes)
    pred_fixes = [pipe_fixes.predict(t)["sentence_sentiment"] for t in texts]
    # Subset masks (test_df may have failure-mode columns from notebook 03)
    negation_fn = lambda t: _neg.has_negation(t)
    conditional_fn = lambda t: _cond.is_conditional(t)
    numeric_fn = lambda t: _num.extract(t) is not None
    hedged_fn = lambda t: _hedge.has_hedge(t)
    res_baseline = evaluate_on_subsets(texts, y_true, pred_baseline, negation_fn=negation_fn, conditional_fn=conditional_fn, numeric_fn=numeric_fn, hedged_fn=hedged_fn)
    res_fixes = evaluate_on_subsets(texts, y_true, pred_fixes, negation_fn=negation_fn, conditional_fn=conditional_fn, numeric_fn=numeric_fn, hedged_fn=hedged_fn)
    print("Baseline (no fixes):")
    print_evaluation_report(res_baseline, "Baseline")
    print("With failure-mode fixes:")
    print_evaluation_report(res_fixes, "With fixes")
    out_dir = ROOT / "data" / "proof" / "credit_risk_sentiment"
    out_dir.mkdir(parents=True, exist_ok=True)
    eval_results = {"baseline": res_baseline, "with_fixes": res_fixes}
    with open(out_dir / "eval_failure_modes.json", "w") as f:
        json.dump(eval_results, f, indent=2)
    print("Saved eval_failure_modes.json to", out_dir)
except Exception as e:
    print("Failure-mode evaluation skipped (install credit_risk.sentiment, spacy):", e)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Baseline (no fixes):
Baseline
--------------------------------------------------
  full: F1 macro = 0.8854 (n = 112)
  negation: F1 macro = 1.0000 (n = 9)
  conditional: F1 macro = 1.0000 (n = 6)
  numeric_comparison: F1 macro = 0.0000 (n = 0)
  hedged: F1 macro = 0.0000 (n = 0)

With failure-mode fixes:
With fixes
--------------------------------------------------
  full: F1 macro = 0.8854 (n = 112)
  negation: F1 macro = 1.0000 (n = 9)
  conditional: F1 macro = 1.0000 (n = 6)
  numeric_comparison: F1 macro = 0.0000 (n = 0)
  hedged: F1 macro = 0.0000 (n = 0)

Saved eval_failure_modes.json to C:\Users\leemi\OneDrive\Desktop\Job_search_2026\ocr-agentic-rag\data\proof\credit_risk_sentiment
